In [ ]:
# -*- coding: utf-8 -*-
"""Fomae: Cross-validation and classification analysis for MSI data."""

from __future__ import absolute_import
from __future__ import division
from __future__ import print_function

import numpy as np
np.random.seed(1337)
import os
import h5py
import matplotlib.pyplot as plt
from sklearn.mixture import GaussianMixture
from sklearn.metrics import mean_squared_error
from scipy import stats
from matplotlib.colors import LinearSegmentedColormap
import matplotlib as mpl
import torch
import pandas as pd
import time
import nibabel as nib
import torch.nn as nn
from Computational_Model_trans import Model_trans

Cd = os.getcwd()
Bd = os.path.dirname(Cd)

plt.rcParams.update(
    {
        "font.family": "serif",
        "font.serif": ["Times New Roman"],
        "mathtext.fontset": "stix",
        "axes.unicode_minus": False,
    }
)

def discrete_cmap(N, base_cmap=None):
    base = plt.cm.get_cmap(base_cmap)
    color_list = base(np.linspace(0, 1, N))
    cmap_name = base.name + str(N)
    return base.from_list(cmap_name, color_list, N)

def Image_Distribution(V,xLoc,yLoc):
    col = max(np.unique(xLoc))
    row = max(np.unique(yLoc))
    Myimg = np.zeros((col,row))
    for i in range(len(xLoc)):
        Myimg[xLoc[i]-1, yLoc[i]-1] = V[i]
    return Myimg

def Correlate_Cluster_MSI(cluster_id,Labels,MSI_D,Peak_Indx,ZCoord_cv,XCoord_cv,YCoord_cv):
    Kimg = Labels==cluster_id
    Kimg = Kimg.astype(int)
    MSI_CleanPeaks = MSI_D[:,Peak_Indx[:]]
    Corr_Val =  np.zeros(len(Peak_Indx))
    
    for i in range(len(Peak_Indx)):
        Corr_Val[i] = stats.pearsonr(Kimg,MSI_CleanPeaks[:,i])[0]
    id_mzCorr = np.argmax(Corr_Val)
    rank_ij =  np.argsort(Corr_Val)[::-1]
    return Corr_Val, rank_ij, MSI_CleanPeaks

def Get_3Dmz_nifti(MSI_CleanPeaks,mz_Peak,XCoord_cv,YCoord_cv,ZCoord_cv,directory):
    mzSections = np.unique(ZCoord_cv)
    Vol_mz = np.zeros((200,200,len(mzSections)))
    nSections = len(mzSections)
    directory_NIFT = directory + '//mz_Vol//Training'
    if not os.path.exists(directory_NIFT):
        os.makedirs(directory_NIFT)
    for Zsec in range(len(mzSections)):
        ij_r = np.argwhere(ZCoord_cv == mzSections[Zsec])
        indx = ij_r[:,0]
        xLoc = XCoord_cv[indx]
        yLoc = YCoord_cv[indx]
        MSI_2D = np.squeeze(MSI_CleanPeaks[indx])
        for idx in range(len(xLoc)):
            Vol_mz[xLoc[idx]-1, yLoc[idx]-1,Zsec] = MSI_2D[idx]  

    I_nii = nib.Nifti1Image(Vol_mz,affine=np.eye(4))
    nib.save(I_nii,directory_NIFT +'//mz_' + str(mz_Peak) + '.nii')

def get_EncFeatures(Latent_z,Train_idx,myZCoord,xLocation,yLocation,directory,order):
    myzSections = np.unique(myZCoord)
    ndim = Latent_z.shape[1]
    for zr in range(len(Train_idx)):
        ij_r = np.argwhere(myZCoord == myzSections[zr])
        indx = ij_r[:,0]
        xLoc = xLocation[indx]
        yLoc = yLocation[indx]
        zSection_Latent_z = np.squeeze(Latent_z[indx,])        
        plt.figure(figsize=(14, 14))
        for j in range(ndim):
            EncFeat = zSection_Latent_z[:,j]
            im = Image_Distribution(EncFeat,xLoc,yLoc);
            ax = plt.subplot(1, ndim, j + 1)    
            plt.imshow(im,cmap="hot");
            ax.get_xaxis().set_visible(False)
            ax.get_yaxis().set_visible(False)
            
        directory_Latz = directory+'//Latent//Training_'+str(order)
        if not os.path.exists(directory_Latz):
            os.makedirs(directory_Latz)
        plt.savefig(directory_Latz + '//EncFetaures_Tissue'+str(myzSections[zr])+'.png',bbox_inches='tight')

def get_gmmImage(Train_idx,Features,nClusters,myZCoord,xLocation,yLocation,directory,order):
    myzSections = np.unique(myZCoord); Zsec=0; 
    C_imgs = np.zeros((200,200,len(range(1,len(Train_idx)+1,1)),nClusters))
    directoryGmm = directory+'//GMM//Training_'+str(order)
    if not os.path.exists(directoryGmm):
        os.makedirs(directoryGmm)
        
    for zr in range(len(Train_idx)):
        im = []
        ij_r = np.argwhere(myZCoord == myzSections[zr])
        indx = ij_r[:,0]
        xLoc = xLocation[indx]
        yLoc = yLocation[indx]
        zSection_labels = Features[indx]
        im = zSection_labels
        im = Image_Distribution(im,xLoc,yLoc)
        MyCmap = discrete_cmap(nClusters, 'jet')    
            
        plt.imshow(im,cmap=MyCmap)
        plt.colorbar(ticks=np.arange(0,nClusters,1))
        plt.axis('off')
        plt.title(f'zr={zr}')
        plt.show()        
        plt.imsave(directoryGmm + '//gmm_Training_'+str(myzSections[zr])+'_K_' + str(nClusters) + '.png',im,cmap=MyCmap)  

        directory_SingleC =directoryGmm + '//GMM_Section_'+str(myzSections[zr])
        if not os.path.exists(directory_SingleC):
            os.makedirs(directory_SingleC)
            
        for c in range(0,nClusters,1):
            cluster_id = c
            Kimg = zSection_labels[:]==cluster_id
            Kimg = Kimg.astype(int)
            for idx in range(len(xLoc)):
                C_imgs[xLoc[idx]-1, yLoc[idx]-1,Zsec,cluster_id] = Kimg[idx]  
            Kimg = Image_Distribution(Kimg,xLoc,yLoc);
            
            segCmp = [MyCmap(0),MyCmap(cluster_id)]
            segCmp[0]= (0,0,0)
            cm = LinearSegmentedColormap.from_list('Walid_cmp',segCmp,N=2)
            plt.imshow(Kimg, cmap=cm);
            plt.colorbar(ticks=np.arange(0,1,1))
            plt.axis('off')
            plt.show()
            plt.imsave(directory_SingleC + '//Cluster_' + str(cluster_id) + '.png',Kimg,cmap=cm)  
        Zsec +=1

    directory_NIFT = directoryGmm + '//NIFTI'
    if not os.path.exists(directory_NIFT):
        os.makedirs(directory_NIFT)
    for c in range(0,nClusters,1):
        I_nii = nib.Nifti1Image(C_imgs[:,:,:,c],affine=np.eye(4))
        nib.save(I_nii,directory_NIFT +'//Label_' + str(c) + '.nii')

def Cluster_To_Nifti(directory,C_imgs,nClusters):
    directory_NIFT = directory + '//GMM_K'+str(nClusters)+'//NIFTI'
    if not os.path.exists(directory_NIFT):
        os.makedirs(directory_NIFT) 
    for c in range(1,nClusters+1,1):
        I_nii = nib.Nifti1Image(C_imgs[:,:,:,c],affine=np.eye(4))
        nib.save(I_nii,directory_NIFT +'//Label_' + str(c) + '.nii')

class GBMClassifier(nn.Module):
    def __init__(self, d_mz, d_model=256, n_head=8, encoder_layers=1):
        super().__init__()
        self.feature_extractor = Model_trans(
            d_mz=d_mz,
            d_model=d_model,
            encoder_layer_num=encoder_layers,
            decoder_layer_num=2,
            n_head=n_head,
            device="cpu",
        )
        self.classifier = nn.Sequential(nn.Linear(d_model, 128), nn.ReLU(), nn.Linear(128, 1))

    def forward(self, x):
        h = self.feature_extractor.backbone(x)
        h = self.feature_extractor.transformerencoder(h)
        return self.classifier(h).squeeze(1)

def load_one_gbm_h5_with_coord(file_path):
    with h5py.File(file_path, "r") as f:
        data_raw = np.array(f["Data"], dtype=np.float32)
        labels = np.array(f["Class_Label"]).squeeze().astype(np.int64)
        x_loc = np.array(f["xLocation"]).squeeze().astype(np.int32)
        y_loc = np.array(f["yLocation"]).squeeze().astype(np.int32)

    if labels.ndim > 1:
        labels = labels.reshape(-1)
    if x_loc.ndim > 1:
        x_loc = x_loc.reshape(-1)
    if y_loc.ndim > 1:
        y_loc = y_loc.reshape(-1)

    if data_raw.shape[1] == labels.shape[0]:
        spectra = data_raw.T
    elif data_raw.shape[0] == labels.shape[0]:
        spectra = data_raw
    else:
        raise ValueError(f"{file_path.name}: Data and labels have mismatched sample counts")

    uniq = set(np.unique(labels).tolist())
    if uniq == {0, 1}:
        labels = labels + 1
    elif uniq == {1, 2}:
        pass
    else:
        raise ValueError(f"{file_path.name}: unsupported label set {uniq}")

    tic_sum = np.sum(spectra, axis=1, keepdims=True)
    valid_mask = tic_sum[:, 0] > 0
    spectra = spectra[valid_mask]
    labels = labels[valid_mask]
    x_loc = x_loc[valid_mask]
    y_loc = y_loc[valid_mask]
    spectra = (spectra / tic_sum[valid_mask]).astype(np.float32)

    return spectra, labels, x_loc, y_loc

In [ ]:
dir = './data/massNet_Raw_h5'
h5_files = sorted([fn for fn in os.listdir(dir) if fn.lower().endswith('.h5')])

if len(h5_files) == 0:
    print("No .h5 files found in directory:", dir)

for filename in h5_files:
    file_path = os.path.join(dir, filename)
    try:
        with h5py.File(file_path, "r") as f:
            class_label = np.array(f["Class_Label"], dtype=np.int32).squeeze()
            xLocation = np.array(f["xLocation"], dtype=np.int32).squeeze()
            yLocation = np.array(f["yLocation"], dtype=np.int32).squeeze()
    except (OSError, KeyError) as e:
        print(f"Skip invalid file: {filename} ({e})")
        continue

    print("File:", os.path.basename(file_path))
    print("Class_Label shape:", class_label.shape)
    print("Unique labels:", np.unique(class_label))

    if len(class_label) == len(xLocation) == len(yLocation):
        h = int(np.max(xLocation))
        w = int(np.max(yLocation))
        label_img = np.full((h, w), np.nan, dtype=np.float32)
        for idx in range(len(class_label)):
            rr = int(xLocation[idx]) - 1
            cc = int(yLocation[idx]) - 1
            label_img[rr, cc] = float(class_label[idx])

        cmap = mpl.cm.get_cmap("magma").copy()
        cmap.set_bad(color="lightgray")

        plt.figure(figsize=(6, 6))
        im = plt.imshow(np.ma.masked_invalid(label_img), cmap=cmap, vmin=1, vmax=2)
        cbar = plt.colorbar(im, ticks=[1, 2])
        cbar.set_ticklabels(["Normal (1)", "Tumor (2)"])
        plt.title("Class_Label Spatial Distribution (Invalid=Gray)")
        plt.axis("off")
        plt.show()
    else:
        print("Class_Label and x/y coordinates have different lengths. Skip spatial plot.")

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import random
import numpy as np
from pathlib import Path
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    precision_recall_fscore_support,
    precision_recall_curve,
    accuracy_score,
    f1_score,
    roc_curve,
    auc,
    confusion_matrix,
    ConfusionMatrixDisplay,
 )
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

SEED = 1337
np.random.seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

GBM_DIR = Path("./data/massNet_Raw_h5")
TEST_FILES = {
    "GBM12_2.h5",
    "GBM22_1.h5",
    "GBM39_1.h5",
}

all_files = sorted([p.name for p in GBM_DIR.glob("*.h5")])
train_files = [f for f in all_files if f not in TEST_FILES]
test_files = [f for f in all_files if f in TEST_FILES]

print("Train files:", train_files)
print("Test files:", test_files)

def load_one_gbm_h5(file_path):
    with h5py.File(file_path, "r") as f:
        data_raw = np.array(f["Data"], dtype=np.float32)
        mz = np.array(f["mzArray"]).squeeze()
        labels = np.array(f["Class_Label"]).squeeze().astype(np.int64)

    if data_raw.shape[0] == len(mz):
        spectra = data_raw.T
    else:
        spectra = data_raw

    if labels.ndim > 1:
        labels = labels.reshape(-1)

    if spectra.shape[0] != labels.shape[0]:
        raise ValueError(f"{file_path.name}: spectra and Class_Label count mismatch: {spectra.shape[0]} vs {labels.shape[0]}")

    tic_sum = np.sum(spectra, axis=1, keepdims=True)
    valid_mask = (tic_sum[:, 0] > 0)
    spectra = spectra[valid_mask]
    labels = labels[valid_mask]
    tic_sum = tic_sum[valid_mask]
    spectra = (spectra / tic_sum).astype(np.float32)

    uniq = set(np.unique(labels).tolist())
    if uniq == {0, 1}:
        labels = labels + 1
    elif uniq == {1, 2}:
        pass
    else:
        raise ValueError(f"{file_path.name}: unsupported label set {uniq}")

    return spectra, labels.astype(np.int64), mz

def build_split(file_list):
    x_parts, y_parts = [], []
    mz_ref = None
    for fname in file_list:
        x_i, y_i, mz_i = load_one_gbm_h5(GBM_DIR / fname)
        if mz_ref is None:
            mz_ref = mz_i
        x_parts.append(x_i)
        y_parts.append(y_i)
        print(f"{fname}: spectra={x_i.shape[0]}, mz_dim={x_i.shape[1]}, labels={dict(zip(*np.unique(y_i, return_counts=True)))}")
    X = np.concatenate(x_parts, axis=0)
    y = np.concatenate(y_parts, axis=0)
    return X, y, mz_ref

X_train_all, y_train_all, mzList = build_split(train_files)
X_test_all, y_test_all, _ = build_split(test_files)

y_train_bin = (y_train_all == 2).astype(np.int64)
y_test_bin = (y_test_all == 2).astype(np.int64)

print("\nTrain set:", X_train_all.shape, "label counts:", dict(zip(*np.unique(y_train_all, return_counts=True))))
print("Test set:", X_test_all.shape, "label counts:", dict(zip(*np.unique(y_test_all, return_counts=True))))

In [ ]:
BATCH_SIZE = 512
D_MODEL = 256
N_HEAD = 8
ENCODER_LAYERS = 1
EPOCHS = 20
LR = 2e-5
WEIGHT_DECAY = 1e-3

model_dir = Path("./Saved_models")
model_dir.mkdir(parents=True, exist_ok=True)


In [ ]:
def evaluate_logits(y_true, y_prob_pos, threshold=0.5):
    y_pred = (y_prob_pos >= threshold).astype(np.int64)
    p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='binary', zero_division=0)
    acc = accuracy_score(y_true, y_pred)
    return p, r, f1, acc, y_pred

def select_precision_weighted_threshold(y_true, y_prob, beta=0.5, min_recall=0.80):
    precisions, recalls, thresholds = precision_recall_curve(y_true, y_prob)
    if len(thresholds) == 0:
        return 0.5, {"precision": 0.0, "recall": 0.0, "f1": 0.0, "fbeta": 0.0}

    precisions = precisions[:-1]
    recalls = recalls[:-1]

    beta2 = beta * beta
    denom = np.clip(beta2 * precisions + recalls, 1e-8, None)
    fbeta = (1 + beta2) * precisions * recalls / denom

    if min_recall is not None:
        fbeta_masked = fbeta.copy()
        fbeta_masked[recalls < min_recall] = -1.0
        if np.max(fbeta_masked) >= 0:
            best_idx = int(np.argmax(fbeta_masked))
        else:
            best_idx = int(np.argmax(fbeta))
    else:
        best_idx = int(np.argmax(fbeta))

    best_th = float(thresholds[best_idx])
    best_p = float(precisions[best_idx])
    best_r = float(recalls[best_idx])
    best_fbeta = float(fbeta[best_idx])
    best_f1 = float((2 * best_p * best_r) / max(best_p + best_r, 1e-8))
    return best_th, {"precision": best_p, "recall": best_r, "f1": best_f1, "fbeta": best_fbeta}

def make_loader(X, y, batch_size, shuffle):
    ds = TensorDataset(
        torch.tensor(X, dtype=torch.float32),
        torch.tensor(y, dtype=torch.long)
    )
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, num_workers=0)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
fold_histories = []
fold_results = []
saved_model_paths = []

oof_prob = np.zeros(len(y_train_bin), dtype=np.float32)
oof_true = y_train_bin.copy()

for fold_id, (tr_idx, val_idx) in enumerate(skf.split(X_train_all, y_train_bin), start=1):
    print(f"\n===== Fold {fold_id}/5 =====")
    X_tr, y_tr = X_train_all[tr_idx], y_train_bin[tr_idx]
    X_val, y_val = X_train_all[val_idx], y_train_bin[val_idx]

    train_loader = make_loader(X_tr, y_tr, BATCH_SIZE, shuffle=True)
    val_loader = make_loader(X_val, y_val, BATCH_SIZE, shuffle=False)

    model = GBMClassifier(
        d_mz=X_train_all.shape[1],
        d_model=D_MODEL,
        n_head=N_HEAD,
        encoder_layers=ENCODER_LAYERS
    ).to(device)

    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    history = {"train_loss": [], "val_loss": []}

    for epoch in range(1, EPOCHS + 1):
        t0 = time.time()
        model.train()
        train_losses = []
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device).float()
            logits = model(xb)
            loss = criterion(logits, yb)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            train_losses.append(loss.item())

        model.eval()
        val_losses = []
        val_probs, val_true = [], []
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device)
                logits = model(xb)
                loss = criterion(logits, yb.float())
                prob_pos = torch.sigmoid(logits)
                val_losses.append(loss.item())
                val_probs.append(prob_pos.cpu().numpy())
                val_true.append(yb.cpu().numpy())

        train_loss = float(np.mean(train_losses))
        val_loss = float(np.mean(val_losses))
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)

        dt = time.time() - t0
        print(f"Fold {fold_id} | Epoch {epoch:02d}/{EPOCHS} | train_loss={train_loss:.6f} | val_loss={val_loss:.6f} | {dt:.1f}s")

    val_prob = np.concatenate(val_probs)
    val_true = np.concatenate(val_true)
    oof_prob[val_idx] = val_prob
    p, r, f1, acc, val_pred = evaluate_logits(val_true, val_prob, threshold=0.5)
    model_name = f"TrainedModel_GBM_trans_fold{fold_id}_F1_{f1:.4f}.pth"
    model_path = model_dir / model_name
    torch.save(model.state_dict(), model_path)
    saved_model_paths.append(str(model_path))

    fold_histories.append(history)
    fold_results.append({
        "fold": fold_id,
        "precision": p,
        "recall": r,
        "f1": f1,
        "accuracy": acc,
        "model_path": str(model_path)
    })

    print(f"Fold {fold_id} validation: Precision={p:.4f}, Recall={r:.4f}, F1={f1:.4f}, Accuracy={acc:.4f}")
    print(f"Model saved: {model_path}")

fold_df = pd.DataFrame(fold_results)
print("\n===== 5-fold validation summary =====")
print(fold_df[["fold", "precision", "recall", "f1", "accuracy", "model_path"]])

best_idx = int(np.argmax(fold_df["f1"].values))
best_fold = int(fold_df.loc[best_idx, "fold"] )
best_model_path = fold_df.loc[best_idx, "model_path"]
print(f"\nBest model from Fold {best_fold}, F1={fold_df.loc[best_idx, 'f1']:.4f}")
print("Best model path:", best_model_path)

DECISION_THRESHOLD, th_metrics = select_precision_weighted_threshold(
    y_true=oof_true,
    y_prob=oof_prob,
    beta=0.5,
    min_recall=0.80
)
print("\n===== OOF threshold search (precision-weighted) =====")
print(f"Selected DECISION_THRESHOLD = {DECISION_THRESHOLD:.4f}")
print(
    "OOF @threshold => "
    f"Precision={th_metrics['precision']:.4f}, "
    f"Recall={th_metrics['recall']:.4f}, "
    f"F1={th_metrics['f1']:.4f}, "
    f"F0.5={th_metrics['fbeta']:.4f}"
)

In [ ]:
best_model_path = "Saved_models/TrainedModel_GBM_trans_fold4_F1_0.9769.pth"
best_model = GBMClassifier(
    d_mz=X_train_all.shape[1],
    d_model=D_MODEL,
    n_head=N_HEAD,
    encoder_layers=ENCODER_LAYERS
).to(device)
best_model.load_state_dict(torch.load(best_model_path, map_location=device))
best_model.eval()

ensemble_models = []
for pth in saved_model_paths:
    m = GBMClassifier(
        d_mz=X_train_all.shape[1],
        d_model=D_MODEL,
        n_head=N_HEAD,
        encoder_layers=ENCODER_LAYERS
    ).to(device)
    m.load_state_dict(torch.load(pth, map_location=device))
    m.eval()
    ensemble_models.append(m)

print(f"Using {len(ensemble_models)} fold models for ensemble inference")
print(f"Decision threshold = {DECISION_THRESHOLD:.4f}")

def infer_split_single(model, X, y_bin, batch_size=512):
    loader = DataLoader(
        TensorDataset(torch.tensor(X, dtype=torch.float32), torch.tensor(y_bin, dtype=torch.long)),
        batch_size=batch_size,
        shuffle=False,
        num_workers=0
    )
    all_prob, all_true = [], []
    t0 = time.time()
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            logits = model(xb)
            prob_pos = torch.sigmoid(logits).cpu().numpy()
            all_prob.append(prob_pos)
            all_true.append(yb.numpy())
    elapsed = time.time() - t0
    y_prob = np.concatenate(all_prob)
    y_true = np.concatenate(all_true)
    return y_true, y_prob, elapsed

def infer_split_ensemble(models, X, y_bin, batch_size=512):
    loader = DataLoader(
        TensorDataset(torch.tensor(X, dtype=torch.float32), torch.tensor(y_bin, dtype=torch.long)),
        batch_size=batch_size,
        shuffle=False,
        num_workers=0
    )
    all_prob, all_true = [], []
    t0 = time.time()
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            fold_probs = []
            for model in models:
                logits = model(xb)
                fold_probs.append(torch.sigmoid(logits).cpu().numpy())
            prob_pos = np.mean(np.stack(fold_probs, axis=0), axis=0)
            all_prob.append(prob_pos)
            all_true.append(yb.numpy())
    elapsed = time.time() - t0
    y_prob = np.concatenate(all_prob)
    y_true = np.concatenate(all_true)
    return y_true, y_prob, elapsed

def build_stats_table(y_true_bin, y_pred_bin, elapsed_seconds):
    precision, recall, f1, support = precision_recall_fscore_support(
        y_true_bin, y_pred_bin, labels=[0, 1], zero_division=0
    )
    acc = accuracy_score(y_true_bin, y_pred_bin)
    table = pd.DataFrame({
        "Class": ["Normal", "Tumor"],
        "Spectra Count": support.astype(int),
        "Precision": np.round(precision, 4),
        "Recall": np.round(recall, 4),
        "F1-score": np.round(f1, 4),
        "Accuracy": [round(acc, 4), round(acc, 4)],
        "Time": [f"{elapsed_seconds:.2f}s", f"{elapsed_seconds:.2f}s"]
    })
    return table

bm_tr_true, bm_tr_prob, bm_tr_time = infer_split_single(best_model, X_train_all, y_train_bin, batch_size=BATCH_SIZE)
bm_tr_pred = (bm_tr_prob >= DECISION_THRESHOLD).astype(np.int64)
best_train_stats_table = build_stats_table(bm_tr_true, bm_tr_pred, bm_tr_time)

bm_te_true, bm_te_prob, bm_te_time = infer_split_single(best_model, X_test_all, y_test_bin, batch_size=BATCH_SIZE)
bm_te_pred = (bm_te_prob >= DECISION_THRESHOLD).astype(np.int64)
best_test_stats_table = build_stats_table(bm_te_true, bm_te_pred, bm_te_time)

print("\n===== Training set statistics (best-model inference) =====")
print(best_train_stats_table.to_string(index=False))

print("\n===== Test set statistics (best-model inference) =====")
print(best_test_stats_table.to_string(index=False))

In [ ]:
plt.figure(figsize=(12, 6))
for i, hist in enumerate(fold_histories, start=1):
    color = plt.cm.tab10(i - 1)
    plt.plot(hist["train_loss"], lw=2, color=color)
    plt.plot(hist["val_loss"], lw=2, linestyle="--", color=color)
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("GBM 5-Fold Train/Validation Loss")
plt.legend(ncol=2, fontsize=9)
plt.tight_layout()
os.makedirs("./pic/GBM", exist_ok=True)
plt.savefig("./pic/GBM/GBM_5Fold_Loss_Curves.png", dpi=300)
plt.show()

fpr_t, tpr_t, _ = roc_curve(bm_te_true, bm_te_prob)
auc_t = auc(fpr_t, tpr_t)

normal_true = 1 - bm_te_true
normal_prob = 1.0 - bm_te_prob
fpr_n, tpr_n, _ = roc_curve(normal_true, normal_prob)
auc_n = auc(fpr_n, tpr_n)

plt.figure(figsize=(7, 6))
plt.plot(fpr_n, tpr_n, lw=3, )
print(f"Test ROC AUC: Tumor={auc_t:.4f}, Normal={auc_n:.4f}")
plt.plot([0, 1], [0, 1], color="gray", lw=1, linestyle="--")
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("GBM Test ROC Curve")
plt.legend(loc="lower right")
plt.tight_layout()
plt.savefig("./pic/GBM/GBM_Test_ROC_Curve.png", dpi=300)
plt.show()

cm = confusion_matrix(bm_te_true, bm_te_pred, labels=[1, 0])
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Tumor", "Normal"])
fig, ax = plt.subplots(figsize=(7, 6))
disp.plot(cmap=plt.cm.Blues, ax=ax, values_format="d", colorbar=False)
ax.set_title("GBM Test Confusion Matrix")
ax.set_xlabel("Predicted Label")
ax.set_ylabel("True Label")
for text in disp.text_.ravel():
    text.set_fontsize(17)
plt.tight_layout()
plt.savefig("./pic/GBM/GBM_Test_Confusion_Matrix.png", dpi=300)
plt.show()

In [ ]:
train_vis_files = train_files
test_vis_files = test_files

def make_background_shape(x_coords, y_coords):
    h = int(np.max(x_coords))
    w = int(np.max(y_coords))
    return h, w

def build_control_and_pred_maps(labels, tumor_prob, x_coords, y_coords):
    h, w = make_background_shape(x_coords, y_coords)

    control_map = np.full((h, w), np.nan, dtype=np.float32)
    pred_map = np.full((h, w), np.nan, dtype=np.float32)

    for idx in range(len(x_coords)):
        rr = int(x_coords[idx]) - 1
        cc = int(y_coords[idx]) - 1
        control_map[rr, cc] = 1.0 if labels[idx] == 2 else 0.0
        pred_map[rr, cc] = float(tumor_prob[idx])

    control_map = np.rot90(np.fliplr(control_map), k=1)
    pred_map = np.rot90(np.fliplr(pred_map), k=1)

    return control_map, pred_map

def _save_single_subplot(img, cmap, norm, out_path, title):
    fig_sub, ax_sub = plt.subplots(figsize=(4, 4))
    ax_sub.imshow(np.ma.masked_invalid(img), cmap=cmap, norm=norm)
    ax_sub.axis("off")
    fig_sub.tight_layout()
    fig_sub.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.close(fig_sub)

def plot_split_maps(file_list, split_name):
    if len(file_list) == 0:
        print(f"{split_name} file list is empty. Skip visualization.")
        return

    blue_purple = LinearSegmentedColormap.from_list(
        "blue_purple",
        ["#0B1D4D", "#2F64B3","#AEADB7", "#7A6BD9", "#B58DFF"],
        N=256
    )
    blue_purple.set_bad(color="#D9DCE5")
    norm = mpl.colors.PowerNorm(gamma=0.7, vmin=0.0, vmax=1.0)

    fig, axes = plt.subplots(len(file_list), 2, figsize=(12, 4 * len(file_list)))
    if len(file_list) == 1:
        axes = np.array([axes])

    safe_split = split_name.replace(" ", "_")
    sub_dir = Path("pic") / "GBM" / f"{safe_split}_subplots"
    sub_dir.mkdir(parents=True, exist_ok=True)

    for i, fname in enumerate(file_list):
        file_path = GBM_DIR / fname
        spectra_i, labels_i, x_i, y_i = load_one_gbm_h5_with_coord(file_path)

        loader_i = DataLoader(
            TensorDataset(torch.tensor(spectra_i, dtype=torch.float32)),
            batch_size=BATCH_SIZE,
            shuffle=False,
            num_workers=0
        )
        tumor_prob_parts = []
        with torch.no_grad():
            for (xb,) in loader_i:
                xb = xb.to(device)
                logits = best_model(xb)
                prob_t = torch.sigmoid(logits).cpu().numpy()
                tumor_prob_parts.append(prob_t)
        tumor_prob = np.concatenate(tumor_prob_parts)

        control_img, pred_img = build_control_and_pred_maps(labels_i, tumor_prob, x_i, y_i)
        title_stem = Path(fname).stem

        ax1 = axes[i, 0]
        ax1.imshow(
            np.ma.masked_invalid(control_img),
            cmap=blue_purple,
            norm=norm
        )
        ax1.set_title(f"{title_stem}: Control (Gray/Normal/Tumor)")
        ax1.axis("off")

        ax2 = axes[i, 1]
        ax2.imshow(
            np.ma.masked_invalid(pred_img),
            cmap=blue_purple,
            norm=norm
        )
        ax2.set_title(f"{title_stem}: Prediction (Gray + Tumor Risk)")
        ax2.axis("off")

        _save_single_subplot(
            control_img,
            blue_purple,
            norm,
            sub_dir / f"{title_stem}_control.png",
            f"{title_stem}: Control"
        )
        _save_single_subplot(
            pred_img,
            blue_purple,
            norm,
            sub_dir / f"{title_stem}_prediction.png",
            f"{title_stem}: Prediction"
        )

    fig.suptitle(f"{split_name} Control vs Prediction Maps", fontsize=14)
    plt.tight_layout(rect=[0, 0, 1, 0.98])
    plt.savefig(f"pic\\GBM\\Test_control_vs_prediction_maps.png", dpi=300, bbox_inches='tight')

    plt.show()

    fig_cb, ax_cb = plt.subplots(figsize=(3, 5))
    sm = mpl.cm.ScalarMappable(norm=norm, cmap=blue_purple)
    sm.set_array([])
    cbar = fig_cb.colorbar(sm, cax=ax_cb)
    cbar.set_label("Shared Scale (0-1)")
    cbar.set_ticks([0.0, 0.5, 1.0])
    fig_cb.suptitle("Colorbar", fontsize=12)
    plt.tight_layout()
    plt.savefig(f"pic\\GBM\\Test_control_vs_prediction_colorbar.png", dpi=300, bbox_inches='tight')
    plt.show()

plot_split_maps(test_vis_files, "Test Split")

In [ ]:
from mpl_toolkits.mplot3d import Axes3D

best_model_path = "Saved_models/TrainedModel_GBM_trans_fold4_F1_0.9769.pth"
vis_model = GBMClassifier(
    d_mz=X_train_all.shape[1],
    d_model=D_MODEL,
    n_head=N_HEAD,
    encoder_layers=ENCODER_LAYERS,
).to(device)
vis_model.load_state_dict(torch.load(best_model_path, map_location=device))
vis_model.eval()

def extract_latent(model, X, batch_size=512):
    loader = DataLoader(
        TensorDataset(torch.tensor(X, dtype=torch.float32)),
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
    )
    feats = []
    with torch.no_grad():
        for (xb,) in loader:
            xb = xb.to(device)
            h = model.feature_extractor.backbone(xb)
            h = model.feature_extractor.transformerencoder(h)
            feats.append(h.cpu().numpy())
    return np.concatenate(feats, axis=0)

X_all = np.concatenate([X_train_all, X_test_all], axis=0)
y_all = np.concatenate([y_train_bin, y_test_bin], axis=0)

latent_all = extract_latent(vis_model, X_all)
print(f"Latent shape: {latent_all.shape}")

import umap

reducer = umap.UMAP(n_components=3, n_neighbors=15, min_dist=0.3, random_state=42)
Z3d = reducer.fit_transform(latent_all)

colors = np.where(y_all == 0, "#2F64B3", "#A14E4E")
alphas = np.where(y_all == 0, 0.35, 0.45)

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection="3d")

for label, name, c in [(0, "Normal", "#2F64B3"), (1, "Tumor", "#A14E4E")]:
    mask = y_all == label
    ax.scatter(
        Z3d[mask, 0],
        Z3d[mask, 1],
        Z3d[mask, 2],
        c=c,
        label=name,
        s=6,
        alpha=0.7,
        edgecolors="none",
    )

ax.set_xticklabels([])
ax.set_yticklabels([])
ax.set_zticklabels([])
ax.view_init(elev=25, azim=135)

plt.tight_layout()
os.makedirs("pic/GBM/latent_viz", exist_ok=True)
plt.savefig("pic/GBM/latent_viz/GBM_latent_3D_UMAP.png", dpi=300, bbox_inches="tight")
plt.show()
print("Saved to pic/GBM/latent_viz/GBM_latent_3D_UMAP.png")
